**In this notebook, We are going to perform Feature engineering, Stratified splitting of data, encoding of categorical data, Standardization, and Normalization.**

**Loading saved Transformed Parquet dataset after cleaning of data (removing missing, duplicates and converting columns into approprirate datatype with construction of target column)**

In [1]:
import os
import pandas as pd
current_dir = os.getcwd()
dataset_candidate = os.path.join(current_dir, "cleaned_ecommerce_orders.parquet")
fallback_path = r"C:\Users\Acer\Desktop\E_Commerce_Return_Prediction\notebooks\cleaned_ecommerce_orders.parquet"
dataset_path = dataset_candidate if os.path.exists(dataset_candidate) else fallback_path
df = pd.read_parquet(dataset_path)
print(f"Primary Dataset loaded from: {dataset_path}")
print(df.head())
print(f"Dataset shape: {df.shape}")


Primary Dataset loaded from: c:\Users\Acer\Desktop\E_Commerce_Return_Prediction\notebooks\cleaned_ecommerce_orders.parquet
        order_id   customer_id order_date product_category     product_id  \
0  ORD0000000000  CUST06438651 2024-12-03          grocery  SKU-GRO-34044   
1  ORD0000000001  CUST04024143 2025-03-19             home  SKU-HOM-22793   
2  ORD0000000003  CUST05454291 2025-07-27      electronics  SKU-ELE-39207   
3  ORD0000000004  CUST04434141 2024-03-11             home  SKU-HOM-09643   
4  ORD0000000005  CUST04504595 2025-02-02         clothing  SKU-CLO-19336   

   quantity  unit_price  order_total payment_method  shipping_cost  \
0         5        4.83        31.59    credit_card           7.44   
1         1      101.25       101.25    credit_card           0.00   
2         4      416.63      1666.52    credit_card           0.00   
3         3      104.94       314.82         paypal           0.00   
4         1       40.34        44.76    credit_card           4.

**Feature Engineering steps**

In [3]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

# 4. Cast boolean flags to integer (Transformation of an feature type to a more suitable type for modeling)
df['discount_applied'] = df['discount_applied'].astype(int)

# Ratio of shipping cost relative to total order value
df['shipping_ratio'] = df['shipping_cost'] / (df['order_total'] + 1e-5)

# Effective unit cost derived from the total charged vs item quantity
df['effective_unit_price'] = df['order_total'] / df['quantity']

# Log transform heavily skewed continuous financial amounts (stabilizes variance)
df['log_order_total'] = np.log1p(df['order_total'])
df['log_unit_price'] = np.log1p(df['unit_price'])

**Define Features and Target (Drop Excluded Columns) Drop the raw target column, intermediate statuses, dates, and ID keys (customer_id, product_id, order_id).**

In [6]:
drop_columns = [
    'order_id',
    'customer_id',
    'product_id',
    'order_date',
    'order_status',
    'return_target',
]

X = df.drop(columns=drop_columns)
y = df['return_target']

### Stratified Splitting (Train, Validation, Test)
**Perform a two-step stratified split to create a 70% Train, 15% Validation, and 15% Test partition while maintaining the 89:11 ratio across all sets.**

In [7]:
# First split: Train (70%) and Temporary (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(
    f'Train: {X_train.shape[0]:,} rows | Minority Class: {y_train.mean():.2%}'
)
print(f'Val:   {X_val.shape[0]:,} rows | Minority Class: {y_val.mean():.2%}')
print(f'Test:  {X_test.shape[0]:,} rows | Minority Class: {y_test.mean():.2%}')

Train: 2,799,317 rows | Minority Class: 11.30%
Val:   599,854 rows | Minority Class: 11.30%
Test:  599,854 rows | Minority Class: 11.30%


### Encoding, Standardization, and Normalization
**Group features by their mathematical properties and process them through a ColumnTransformer:**

**Categorical Encoding: One-Hot Encode product_category and payment_method with handle_unknown='ignore' and drop='first'.**

**Standardization / Scaling: Use StandardScaler (zero mean, unit variance) or RobustScaler (median and IQR to handle residual price/shipping outliers).**

**Binary Passthrough: Keep discount_applied unchanged.**

In [8]:
# Identify feature groups
categorical_cols = ['product_category', 'payment_method']
numerical_cols = [
    'quantity',
    'unit_price',
    'order_total',
    'shipping_cost',
    'shipping_ratio',
    'effective_unit_price',
    'log_order_total',
    'log_unit_price',
]
binary_cols = ['discount_applied']

# Build ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_cols,
        ),  # Change to RobustScaler() if extreme outliers remain
        (
            'cat',
            OneHotEncoder(
                drop='first', sparse_output=False, handle_unknown='ignore'
            ),
            categorical_cols,
        ),
        ('passthrough', 'passthrough', binary_cols),
    ],
    verbose_feature_names_out=False,
)


**Fit ONLY on Training Data to prevent leakage; Transform Val & Test**

In [9]:
# Fit ONLY on Training Data to prevent leakage; Transform Val & Test
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Extract column names for inspection
feature_names = preprocessor.get_feature_names_out()

# Convert back to DataFrame (Optional, preserves feature names for model introspection)
X_train_final = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_val_final = pd.DataFrame(
    X_val_processed, columns=feature_names, index=X_val.index
)
X_test_final = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

print(f'\nFinal feature count: {X_train_final.shape[1]}')
print(f'Final training matrix shape: {X_train_final.shape}')


Final feature count: 20
Final training matrix shape: (2799317, 20)


### Derive Model Cost Weights from the Train Split
**Calculate the exact scale weight directly from y_train for cost-sensitive training in XGBoost or LightGBM:**

In [10]:
# Compute scale_pos_weight = count(negative) / count(positive)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Calculated scale_pos_weight for XGBoost/LightGBM: {scale_pos_weight:.2f}')

Calculated scale_pos_weight for XGBoost/LightGBM: 7.85


**Saving all splits into parquet and joblib files for future reusability and modularity of code**

In [11]:
# Save features (DataFrames)
X_train_final.to_parquet("X_train.parquet", index=False)
X_val_final.to_parquet("X_val.parquet", index=False)
X_test_final.to_parquet("X_test.parquet", index=False)

# Save targets (Series to Parquet)
y_train.to_frame().to_parquet("y_train.parquet", index=False)
y_val.to_frame().to_parquet("y_val.parquet", index=False)
y_test.to_frame().to_parquet("y_test.parquet", index=False)

# Also save the fitted preprocessor so you can transform future inference data
import joblib

joblib.dump(preprocessor, "preprocessor.joblib")
print("All splits and preprocessor saved successfully!")

All splits and preprocessor saved successfully!
